In [45]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import scipy as sp
from sympy import *
%matplotlib ipympl 



In [46]:
delta = 8/9

def A(x):
	return np.sqrt(1-delta**2*(1/np.cosh(x))**2)

r_minus_max = -A(0)
r_plus_min = A(0)

r_minus_max, r_plus_min

(-0.45812284729085123, 0.45812284729085123)

In [47]:
X, Y = np.meshgrid(np.linspace(-2,2,100), np.linspace(-2,2,100))
Z = X + 1j*Y

In [58]:
def integrand(eta,endpts,p, branch:str):
	alpha = endpts[0]
	beta = endpts[1]
	if branch=='neg':
		# return eta**p*np.sqrt(eta-alpha)/np.sqrt(-(1-eta**2)*(eta-beta))*(p+1 + (eta*(3*eta**2-2*beta*eta-1)/(2*(1-eta**2)*(eta-beta))))
		return np.sqrt(eta-alpha)*( ((p+1)*eta**p)/np.sqrt(-(1-eta**2)*(eta - beta)) - eta**(p+1)*(3*eta**2-2*beta*eta-1)/(2*((1-eta**2)*(beta-eta))**(3/2) ))
	
	if branch=='pos':
		# return eta**p*np.sqrt(beta-eta)/np.sqrt((1-eta**2)*(eta-alpha))*(p+1 + (eta*(3*eta**2-2*alpha*eta-1)/(2*(1-eta**2)*(eta-alpha))))
		return np.sqrt(beta - eta)*( ((p+1)*eta**p)/np.sqrt((1-eta**2)*(eta - alpha)) - eta**(p+1)*(-3*eta**2+2*alpha*eta+1)/(2*((1-eta**2)*(eta-alpha))**(3/2)))
	else: 
		return ValueError('Need to specify branch.')


def total_moments(endpts:list,p,x,t, error=False):
	alpha = endpts[0]
	beta = endpts[1]
	
	minus_moment = r_minus_max**(p+1)*np.sqrt(r_minus_max - alpha)/np.sqrt(-(1-r_minus_max**2)*(r_minus_max-beta)) - sp.integrate.quad(integrand, alpha,r_minus_max, args=(endpts,p,'neg'),points=alpha)[0]
	plus_moment = -r_plus_min**(p+1)*np.sqrt(beta-r_plus_min)/np.sqrt((1-r_plus_min**2)*(r_plus_min-alpha)) - sp.integrate.quad(integrand, r_plus_min,beta, args=(endpts,p,'pos'), points=beta)[0]
	sum_of_moments = minus_moment + plus_moment

	if p==0:
		result = sum_of_moments + (x+(alpha+beta)*t)
	
	if p==1:
		result = sum_of_moments + (1/2)*((alpha+beta)*x+((3/2)*alpha**2+(3/2)*beta**2+alpha*beta)*t)

	if error==True:
		print(f"The error from numerically integrating the p={p} moment is {sum_of_moments[1]}.")
	
	return result
	# return np.array([np.real(result),np.imag(result).imag])



def total_moments_system(endpts,x,t, error=False):
	# alpha = endpts[0]+1j*endpts[1]
	# beta = endpts[2]+1j*endpts[3]
	# realmatrix = np.array([np.real(total_moments([alpha,beta],0,x,t)), np.real(total_moments([alpha,beta],1,x,t))])
	# imagmatrix = np.array([np.imag(total_moments([alpha,beta],0,x,t)), np.imag(total_moments([alpha,beta],1,x,t))])
	# return np.concatenate([realmatrix,imagmatrix])
	matrix = np.array([total_moments(endpts,0,x,t), total_moments(endpts,1,x,t)])
	return matrix



In [56]:
x_window = 0
t_window = 5
X = np.linspace(0, x_window, 20)
T = np.linspace(0, t_window, 20)

array([0., 0.])

In [75]:
# testing for t=0 at
x0 = 3
t0= 0.0
alpha = -A(x0)
beta = A(x0)
endpts = [alpha,beta]
# endpts,total_moments_system([alpha,beta],x0,t0)
# sp.optimize.newton(total_moments_system, [-A(x0),A(x0)], args=(x0,t0))

endpts, sp.optimize.newton(total_moments, [-A(x0),A(x0)], args=(0,x0,t0)), sp.optimize.newton(total_moments, [-A(x0),A(x0)], args=(1,x0,t0))


/Users/JoanneDong/anaconda3/lib/python3.11/site-packages/scipy/optimize/_zeros_py.py:465: RuntimeWarning: RMS of 9.6381e-06 reached
  warnings.warn(


([-0.9960946805449884, 0.9960946805449884],
 array([-0.99609468,  0.99609468]),
 array([-0.9961015,  0.9961015]))

In [68]:
# sp.optimize.newton(total_moments_system, [-A(x0),A(x0)], args=(x0,t0))
sp.optimize.newton(total_moments, [-A(x0),A(x0)], args=(0,x0,t0)), sp.optimize.newton(total_moments, [-A(x0),A(x0)], args=(1,x0,t0))


/Users/JoanneDong/anaconda3/lib/python3.11/site-packages/scipy/optimize/_zeros_py.py:465: RuntimeWarning: RMS of 7.04051e-06 reached
  warnings.warn(


(array([-0.45812285,  0.45812285]), array([-0.45812783,  0.45812783]))

TypeError: 'tuple' object is not callable